# IDRiD batch 1 — one-time cross-dataset evaluation of the frozen E1 and E2 graders (record §65)

**Evaluation only. Nothing is trained, tuned or selected.** Everything that defines the run is in
`idrid_batch1_protocol.json` (six E1 / E2 checkpoint hashes, the three stored P tables, the comparisons, the disclosures)
and, unchanged, in its parent `idrid_grading_protocol.json` (the 103 images and labels by hash, the three images excluded
from the primary set, Stage-2 preprocessing, the 512 RGB frame, T4 / mixed_float16 / batch 8, tolerances, bootstrap).

- **P is not run.** Its per-image predictions from the earlier one-time run are reused after their hashes are checked.
- **E1 and E2 take RGB only**: Stage 3 and Stage 4 are not run and no cache of theirs is read.
- Primary set: 100 images (`IDRiD_088`, `IDRiD_089`, `IDRiD_091` excluded). Sensitivity: all 103.

Order of the cells — it cannot be changed:
1. Setup.
2. Protocol and pinned files (six checkpoints, three P tables).
3. **Pre-inference gates.** Model construction and precision; APTOS parity for E1 and E2 (ten raw images through Stage 2 and
   the resize; all 730 validation images against the stored tables). If a gate fails, stop: do not run cell 4.
4. **The IDRiD run, once.** Refuses without the confirmation token, without passed gates from this runtime, or if the lock exists.
5. Print the reports.

GPU runtime (T4). **Disclosures that travel with the result:** IDRiD was used once before for an earlier pipeline; E1 / E2 were
not tuned or selected on it; Stage 4 (E1's teacher) was trained on IDRiD's camera domain. This is a cross-dataset evaluation of
models fixed beforehand, not an untouched project-level test.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad
import idrid_batch1_eval as eb, idrid_grading_eval as ig
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] PROTOCOL + PINNED FILES (no IDRiD image is read) ====
PROTOCOL, PROTOCOL_SHA, PARENT = eb.load_protocol()
DRIVE_ROOT = colab_config.DRIVE.project_root
EXP = colab_config.DRIVE.experiments_root
OUT_ROOT = posixpath.join(EXP, "IDRiDGrading")
assert tf.config.list_physical_devices("GPU"), "GPU runtime required (the locked settings are the T4 settings)"
FILES = eb.verify_files(PROTOCOL, DRIVE_ROOT)        # SHA-256 of the six checkpoints and the three P tables; BEST slots
print("batch protocol", PROTOCOL_SHA, "| parent", PROTOCOL["parent_protocol"]["sha256"][:12], "| commit", REPO_COMMIT)
for _name, _entry in PROTOCOL["checkpoints"].items():
    print(f"  {_name:<12} {_entry['sha256']}  {_entry['best_slot']} (epoch index {_entry['best_epoch_index']})")
for _s in eb.SEEDS:
    print(f"  P seed {_s:<5} stored table {PROTOCOL['stored_p_tables'][f'p_seed{_s}']['sha256']}  (reused, not run)")
print("primary exclusions:", PARENT["dataset"]["primary_exclusions"], "| inference:", PARENT["inference"])
print("primary comparisons:", PROTOCOL["comparisons"]["primary"], "| descriptive:", PROTOCOL["comparisons"]["secondary_descriptive"])
print("lock present:", os.path.exists(eb.lock_path(OUT_ROOT, PROTOCOL_SHA)))

In [ ]:
# ==== [3] PRE-INFERENCE GATES -- must pass before any IDRiD image is read ====
GEN4 = cache.stage4_generation_id(PARENT["checkpoints"]["stage4_unet"]["sha256"], len(v2cfg.STAGE4_V2A_CLASSES))
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=PARENT["checkpoints"]["stage4_unet"]["sha256"])   # cached APTOS RGB frames, on Drive
GATE = eb.gates(DRIVE_ROOT, posixpath.join(colab_config.APTOS2019_RAW_DIR, "train_images"), BUNDLE,
                posixpath.join(OUT_ROOT, f"batch1_gates_{PROTOCOL_SHA[:12]}"),
                aptos_processed_dir=posixpath.join(colab_config.APTOS2019_PROCESSED_DIR, "train_images"))
print(json.dumps({"PASS": GATE["PASS"], "failures": GATE["failures"],
                  "grading_output_dtype": GATE["checks"]["grading_output_dtype"],
                  "raw_subset_images": GATE["checks"]["raw_subset"]["image_checks"],
                  "raw_subset_predictions": GATE["checks"]["raw_subset"]["predictions"],
                  "full_validation_730": GATE["checks"]["full_validation"],
                  "idrid_images_read": GATE["idrid_images_read"], "environment": GATE["environment"]}, indent=1, default=float))
assert GATE["PASS"], "GATES FAILED -- stop here. Do not run the IDRiD cell. Report the failures."


In [ ]:
# ==== [4] THE ONE-TIME IDRiD BATCH-1 RUN (E1 and E2; P reused) ====
# Runs once. To run it, set CONFIRM to the token printed below. A second run is refused; after a technical failure only,
# set TECHNICAL_RERUN_REASON to a written reason (it is recorded with the run).
print("token:", PROTOCOL["confirmation_token"])
CONFIRM = None
TECHNICAL_RERUN_REASON = None
RESULT = eb.run(DRIVE_ROOT, colab_config.IDRID_GRADING_RAW_DIR, OUT_ROOT, GATE, confirm=CONFIRM,
                technical_rerun_reason=TECHNICAL_RERUN_REASON,
                command="colab/notebooks/idrid_batch1_e1e2.ipynb cell 4")

In [ ]:
# ==== [5] REPORTS (reads what cell 4 wrote) ====
print(open(posixpath.join(RESULT["out_dir"], "primary_report_100.md"), encoding="utf-8").read())
print(open(posixpath.join(RESULT["out_dir"], "sensitivity_report_103.md"), encoding="utf-8").read())
print("output:", RESULT["out_dir"])